## Feature Engineering for Time Series with External Factors
## Objective
In this notebook, we create new features that help machine learning
models understand time dependencies and seasonal behavior in electricity demand

In [65]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import math
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [66]:
ts=pd.read_csv(r"C:\Users\Chinnarajan M\OneDrive\Documents\Chinna projects\time_series_with_external_factors.csv")
ts['date']=pd.to_datetime(ts['date'])
ts=ts.set_index('date')

In [67]:
ts.head()

,electricity_demand,temperature_celsius,rainfall_mm,is_holiday
date,,,,
2019-01-01,414.34,26.17,1.85,0
2019-01-02,399.36,25.07,12.70,0
2019-01-03,391.19,26.81,8.28,0
2019-01-04,433.55,28.73,5.96,0
2019-01-05,428.43,25.39,2.98,1


Time series ML models cannot automatically understand past behavior.
We must explicitly create features such as:
- Lag values
- Rolling statistics
- Calendar information

In [68]:
# Create Lag Features
ts['lag_1'] = ts['electricity_demand'].shift(1)
ts['lag_7'] = ts['electricity_demand'].shift(7)
ts['lag_30'] = ts['electricity_demand'].shift(30)

ts[['electricity_demand','lag_1','lag_7','lag_30']].head(35)

,electricity_demand,lag_1,lag_7,lag_30
date,,,,
2019-01-01,414.34,NaN,NaN,NaN
2019-01-02,399.36,414.34,NaN,NaN
2019-01-03,391.19,399.36,NaN,NaN
2019-01-04,433.55,391.19,NaN,NaN
2019-01-05,428.43,433.55,NaN,NaN
2019-01-06,417.23,428.43,NaN,NaN
2019-01-07,443.44,417.23,NaN,NaN
2019-01-08,399.71,443.44,414.34,NaN
2019-01-09,408.87,399.71,399.36,NaN


Feature      Meaning

lag_1 -     Yesterday’s demand

lag_7 -     Demand one week ago

lag_30 -    Monthly pattern


In [69]:
# Creating Rolling Mean Features
ts['rolling_7_mean'] = ts['electricity_demand'].rolling(window=7).mean()
ts['rolling_30_mean'] = ts['electricity_demand'].rolling(window=30).mean()

ts[['rolling_7_mean','rolling_30_mean']].head(40)

,rolling_7_mean,rolling_30_mean
date,,
2019-01-01,NaN,NaN
2019-01-02,NaN,NaN
2019-01-03,NaN,NaN
2019-01-04,NaN,NaN
2019-01-05,NaN,NaN
2019-01-06,NaN,NaN
2019-01-07,418.220000,NaN
2019-01-08,416.130000,NaN
2019-01-09,417.488571,NaN


Why Rolling Mean ?
Smooths noise
Captures short & long-term trends

In [70]:
# Create Calendar Features
ts['day'] = ts.index.day
ts['month'] = ts.index.month
ts['day_of_week'] = ts.index.dayofweek
ts['week_of_year'] = ts.index.isocalendar().week

ts[['day','month','day_of_week','week_of_year']].head()

,day,month,day_of_week,week_of_year
date,,,,
2019-01-01,1,1,1,1
2019-01-02,2,1,2,1
2019-01-03,3,1,3,1
2019-01-04,4,1,4,1
2019-01-05,5,1,5,1


Explanation
day_of_week: weekday vs weekend behavior
month: seasonal pattern
week_of_year: yearly cycles

In [71]:
# Create Weekend Feature
ts['is_weekend'] = np.where(ts['day_of_week'] >= 5, 1, 0)
ts[['day_of_week','is_weekend']].head(10)

,day_of_week,is_weekend
date,,
2019-01-01,1,0
2019-01-02,2,0
2019-01-03,3,0
2019-01-04,4,0
2019-01-05,5,1
2019-01-06,6,1
2019-01-07,0,0
2019-01-08,1,0
2019-01-09,2,0


Insight
Weekends often behave differently from weekdays

In [72]:
# Handle Missing Values
ts.isnull().sum()

electricity_demand      0
temperature_celsius     0
rainfall_mm             0
is_holiday              0
lag_1                   1
lag_7                   7
lag_30                 30
rolling_7_mean          6
rolling_30_mean        29
day                     0
month                   0
day_of_week             0
week_of_year            0
is_weekend              0
dtype: int64

In [73]:
# Drop Missing Rows
ts=ts.dropna()
ts.isna().sum()

electricity_demand     0
temperature_celsius    0
rainfall_mm            0
is_holiday             0
lag_1                  0
lag_7                  0
lag_30                 0
rolling_7_mean         0
rolling_30_mean        0
day                    0
month                  0
day_of_week            0
week_of_year           0
is_weekend             0
dtype: int64

In [74]:
# Feature vs Target Separation
X = ts.drop('electricity_demand', axis=1)
Y = ts['electricity_demand']

print("Feature shape:", X.shape)
print("Target shape:", Y.shape)

Feature shape: (1796, 13)
Target shape: (1796,)


In [75]:
X # it will contain only Feature variables

,temperature_celsius,rainfall_mm,is_holiday,lag_1,lag_7,lag_30,rolling_7_mean,rolling_30_mean,day,month,day_of_week,week_of_year,is_weekend
date,,,,,,,,,,,,,
2019-01-31,28.88,3.04,0,445.62,430.58,414.34,435.101429,426.843667,31,1,3,5,0
2019-02-01,33.94,2.47,0,429.01,436.01,399.36,437.488571,428.622333,1,2,4,5,0
2019-02-02,30.35,6.83,1,452.72,438.60,391.19,435.014286,429.625333,2,2,5,5,1
2019-02-03,28.41,7.43,1,421.28,411.02,433.55,439.692857,429.966000,3,2,6,5,1
2019-02-04,32.31,5.02,0,443.77,441.64,428.43,444.185714,431.454667,4,2,0,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-27,24.47,10.17,0,422.51,393.25,392.15,406.455714,399.983000,27,12,2,52,0
2023-12-28,24.03,3.58,0,405.44,405.71,361.87,407.762857,401.749333,28,12,3,52,0
2023-12-29,23.36,4.53,0,414.86,384.14,403.54,409.514286,401.511333,29,12,4,52,0


In [76]:
Y # it will contain only Target variables

date
2019-01-31    429.01
2019-02-01    452.72
2019-02-02    421.28
2019-02-03    443.77
2019-02-04    473.09
               ...  
2023-12-27    405.44
2023-12-28    414.86
2023-12-29    396.40
2023-12-30    466.54
2023-12-31    402.21
Name: electricity_demand, Length: 1796, dtype: float64

In [77]:
X.columns # Final Feature List

Index(['temperature_celsius', 'rainfall_mm', 'is_holiday', 'lag_1', 'lag_7',
       'lag_30', 'rolling_7_mean', 'rolling_30_mean', 'day', 'month',
       'day_of_week', 'week_of_year', 'is_weekend'],
      dtype='object')

Final Features Created
Lag features
Rolling statistics
Weather variables
Holiday & weekend flags
Calendar features

In [78]:
# Save Feature_Engineered Dataset
ts.to_csv("feature_engineered_time_series.csv")

# Key Learnings from Notebook-2
1. Lag features capture historical dependency
2. Rolling features smooth volatility
3. Calendar features capture seasonality
4. Feature engineering improves ML performance drastically